# Camada Gold Modelo Analítico (Star Schema)

**Pipeline:** Análise da Alfabetização no Brasil
**Responsabilidade:** ler a Silver limpa, integrar as fontes e modelar o star schema para consumo do Power BI.

**Modelo:**
- `dim_uf` — dimensão de estados
- `dim_municipio` — dimensão de municípios
- `dim_tempo` — dimensão de anos
- `fato_indicador` — fato central: taxa realizada vs. meta, particionado por ano

**Princípio da camada:** a Gold é a única camada que o Power BI acessa — sem transformação pesada no Power Query, só leitura direta.

## 1. Configuração de Acesso

Conexão ao ADLS Gen2 via **Azure Key Vault**, mesmo padrão da Bronze e da Silver.

In [ ]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# ============================================================
from notebookutils import mssparkutils
import os

CONTA_STORAGE = os.getenv("STORAGE_ACCOUNT_NAME", "stalfalfabetizacao")
KEY_VAULT_URL = os.getenv("KEY_VAULT_URL", "https://kv-alfabetizacao.vault.azure.net/")
KEY_VAULT_SECRET_NAME = os.getenv("KEY_VAULT_SECRET_NAME", "storage-account-key")

CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    KEY_VAULT_URL,
    KEY_VAULT_SECRET_NAME
)

spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

CAMINHO_SILVER = f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_GOLD   = f"abfss://gold@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")

StatementMeta(sparkpool, 11, 7, Finished, Available, Finished, False)

Conexão configurada com sucesso!


## 2. Leitura da Camada Silver

A Gold nunca lê da Bronze — sempre parte da Silver já tratada e padronizada.

In [11]:
# ============================================================
# Leitura das tabelas Silver
# A Gold nunca lê da Bronze — sempre parte da Silver limpa
# ============================================================

df_uf            = spark.read.parquet(CAMINHO_SILVER + "uf/")
df_municipio     = spark.read.parquet(CAMINHO_SILVER + "municipio/")
df_meta_brasil   = spark.read.parquet(CAMINHO_SILVER + "meta_brasil/")
df_meta_uf       = spark.read.parquet(CAMINHO_SILVER + "meta_por_uf/")
df_meta_municipio = spark.read.parquet(CAMINHO_SILVER + "meta_por_municipio/")
df_ts_municipio  = spark.read.parquet(CAMINHO_SILVER + "ts_municipio/")
df_ts_estado     = spark.read.parquet(CAMINHO_SILVER + "ts_estado/")

print("Silver lida com sucesso!")
print(f"  municipio:      {df_municipio.count()} linhas")
print(f"  meta_municipio: {df_meta_municipio.count()} linhas")
print(f"  ts_municipio:   {df_ts_municipio.count()} linhas")

StatementMeta(sparkpool, 11, 8, Finished, Available, Finished, False)

Silver lida com sucesso!
  municipio:      23995 linhas
  meta_municipio: 10704 linhas
  ts_municipio:   12416 linhas


## 3. Construção do Star Schema

Modelagem dimensional: `dim_uf`, `dim_municipio`, `dim_tempo` e o fato central `fato_indicador`, que integra o realizado (`municipio`) com a meta (`meta_por_municipio`) via join.

In [12]:
# ============================================================
# GOLD — Star Schema (corrigido — qualificando colunas ambíguas)
# ============================================================
from pyspark.sql.functions import col

# ── DIM_UF ──────────────────────────────────────────────────
dim_uf = df_uf \
    .select("sigla_uf") \
    .dropDuplicates() \
    .orderBy("sigla_uf")

# ── DIM_MUNICÍPIO ────────────────────────────────────────────
dim_municipio = df_municipio \
    .select("id_municipio") \
    .dropDuplicates(["id_municipio"]) \
    .join(
        df_ts_municipio.select(
            col("id_municipio"),
            col("sigla_uf")
        ).dropDuplicates(["id_municipio"]),
        on="id_municipio",
        how="left"
    ) \
    .orderBy("id_municipio")

# ── DIM_TEMPO ────────────────────────────────────────────────
dim_tempo = df_municipio \
    .select("ano") \
    .dropDuplicates() \
    .orderBy("ano")

# ── FATO_INDICADOR ───────────────────────────────────────────
# Alias nos DataFrames antes do join para evitar ambiguidade —
# ambos têm coluna "taxa_alfabetizacao" e o Spark não saberia
# qual usar no select sem a qualificação "mun." / "meta."
df_mun = df_municipio.alias("mun")
df_meta = df_meta_municipio.alias("meta")

fato_indicador = df_mun \
    .join(
        df_meta,
        on=["id_municipio", "ano", "rede"],
        how="left"
    ) \
    .select(
        # Colunas qualificadas com alias para resolver ambiguidade pós-join
        col("mun.id_municipio"),
        col("mun.ano"),
        col("mun.rede"),
        col("mun.taxa_alfabetizacao").alias("taxa_realizada"),
        col("mun.media_portugues"),
        col("meta.taxa_alfabetizacao").alias("taxa_meta_referencia"),
        col("meta.meta_alfabetizacao_2024"),
        col("meta.meta_alfabetizacao_2025"),
        col("meta.meta_alfabetizacao_2026"),
        col("meta.meta_alfabetizacao_2027"),
        col("meta.meta_alfabetizacao_2028"),
        col("meta.meta_alfabetizacao_2029"),
        col("meta.meta_alfabetizacao_2030"),
        col("meta.nivel_alfabetizacao"),
        col("meta.percentual_participacao")
    ) \
    .dropDuplicates()

# Confirma
print("Star schema construído:")
print(f"  dim_uf:         {dim_uf.count()} linhas")
print(f"  dim_municipio:  {dim_municipio.count()} linhas")
print(f"  dim_tempo:      {dim_tempo.count()} linhas")
print(f"  fato_indicador: {fato_indicador.count()} linhas")

StatementMeta(sparkpool, 11, 9, Finished, Available, Finished, False)

Star schema construído:
  dim_uf:         25 linhas
  dim_municipio:  5550 linhas
  dim_tempo:      2 linhas
  fato_indicador: 23995 linhas


## 4. Gravação da Camada Gold

Persistência em Parquet, com `fato_indicador` **particionado por ano** — otimização de custo: queries filtrando por ano leem só a partição necessária (FinOps).

In [13]:
# ============================================================
# Gravação da camada Gold em formato Parquet
# Particionamento por "ano" na fato_indicador —
# queries filtrando por ano leem só a partição necessária
# (menos dados lidos = menor custo = FinOps)
# ============================================================

dim_uf.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "dim_uf/"
)
dim_municipio.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "dim_municipio/"
)
dim_tempo.write.mode("overwrite").parquet(
    CAMINHO_GOLD + "dim_tempo/"
)

# Fato particionado por ano — padrão FinOps
fato_indicador.write \
    .mode("overwrite") \
    .partitionBy("ano") \
    .parquet(CAMINHO_GOLD + "fato_indicador/")

print("Gold gravada com sucesso!")
print("Tabelas disponíveis:")
print(f"  {CAMINHO_GOLD}dim_uf/")
print(f"  {CAMINHO_GOLD}dim_municipio/")
print(f"  {CAMINHO_GOLD}dim_tempo/")
print(f"  {CAMINHO_GOLD}fato_indicador/  (particionada por ano)")

StatementMeta(sparkpool, 11, 10, Finished, Available, Finished, False)

Gold gravada com sucesso!
Tabelas disponíveis:
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/dim_uf/
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/dim_municipio/
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/dim_tempo/
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/fato_indicador/  (particionada por ano)


### Validação da Gold

In [14]:
# ============================================================
# Validação da Gold
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.functions import col
# Lê de volta do storage pra confirmar que gravou certo
fato_lido = spark.read.parquet(CAMINHO_GOLD + "fato_indicador/")

print("=== VALIDAÇÃO FATO_INDICADOR ===")
print(f"Total linhas:     {fato_lido.count()}")
print(f"Nulos id_municipio: {fato_lido.filter(col('id_municipio').isNull()).count()}")
print(f"Nulos taxa_realizada: {fato_lido.filter(col('taxa_realizada').isNull()).count()}")
print(f"Anos disponíveis:")
fato_lido.select("ano").dropDuplicates().orderBy("ano").show()
print(f"Taxa min/max:")
fato_lido.select(
    F.min("taxa_realizada"),
    F.max("taxa_realizada")
).show()

StatementMeta(sparkpool, 11, 11, Finished, Available, Finished, False)

=== VALIDAÇÃO FATO_INDICADOR ===
Total linhas:     23995
Nulos id_municipio: 0
Nulos taxa_realizada: 0
Anos disponíveis:
+----+
| ano|
+----+
|2023|
|2024|
+----+

Taxa min/max:
+-------------------+-------------------+
|min(taxa_realizada)|max(taxa_realizada)|
+-------------------+-------------------+
|               2.12|              100.0|
+-------------------+-------------------+

